In [1]:
import pandas as pd
import numpy as np
# import the matplotlib.pyplot library and name it as plt
import matplotlib.pyplot as plt
# #import the PdfPages to save all plots

from matplotlib.backends.backend_pdf import PdfPages
import math
from scipy import stats

In [2]:
#home_id
patients=[1135,1450,1497,1498,1504,1506,1508,1507,1511,1514,1526,1540,1590,1615,1618,1693,1464,1522,1603,1609,1610,1607,1627,1628,1657,1660,1665,1724,1542,1541,1559,1571,1572,1582,1586,1591,1592,1600,1617,1699,1706,1707,1754]

In [5]:
def read_clinical():
    clinical = pd.read_csv('filtered_clinical.csv', index_col=0)
    # Create a new order for the columns
    new_columns = ['sub_id']+['home_id'] +['study'] +[col for col in clinical.columns if (col != 'home_id' and col !='sub_id' and col != 'study')]
    # Reorder the DataFrame
    clinical = clinical[new_columns]
    
    
    return clinical

In [4]:
def readData(patient):
    tm = pd.read_csv(f"{patient}_cleaned_alldata"'.csv')
    out = np.array(tm).tolist()  
    
    for item in out:
        if 'Bathroom' in item[1]:
            item[1]='Bathroom'
            
        if item[1]=='Sensor Line' or item[1]=='Extra sensor line' or item[1]=='Hallway 1'or item[1]=='Hallway 2'or item[1]=='Hallway 3'or item[1]=='Hallway 4':
            item[1]='Hallway'
            
        if 'Entrance Hallway' in item[1]:
            item[1]='Entrance Hallway'
            
        if 'Living Room' in item[1]:
            item[1]='Living Room'
            
        if 'Garage' in item[1]:
            item[1]='Garage Door'
    return out

In [6]:
def find_date(out):
    data_by_date = {}
    for entry in out:
        timestamp, location = entry
        date = timestamp.split(" ")[0]
        
        if date not in data_by_date:
            data_by_date[date] = []
    return data_by_date

In [7]:
def remove_covid(out):
    # Remove data after 2020.03
    filtered_data = []
    cutoff_date = datetime(2020, 3, 1)

    for entry in out:
        date_time = datetime.strptime(entry[0], "%d/%m/%Y %H:%M:%S")
        if date_time >= cutoff_date:
            break
        filtered_data.append(entry)
    return filtered_data

In [8]:
from datetime import datetime

def remove_day_optimized(out):
    # Use a dictionary to group data by date and collect unique sensor activations and counts
    data_by_date = {}
    
    for entry in out:
        timestamp, location = entry
        date = timestamp.split(" ")[0]
        
        if date not in data_by_date:
            data_by_date[date] = {'devices': set()}
        
        
        # Collect unique sensor locations per day
        data_by_date[date]['devices'].add(location)

    # Determine which dates to remove based on criteria
    remove_dates = set()
    for date, info in data_by_date.items():
        if not info['devices'] or 'Bathroom' not in info['devices'] or len(info['devices']) == 1:
            print(date, info)
            remove_dates.add(date)

    # Filter out entries from days that meet removal criteria
    filtered_out = [entry for entry in out if entry[0].split(" ")[0] not in remove_dates]

    # Update and sort the list of unique dates excluding the removed ones
    new_dates = sorted([date for date in data_by_date if date not in remove_dates],
                       key=lambda x: datetime.strptime(x, "%d/%m/%Y"))
    
    return filtered_out, new_dates

In [9]:
from datetime import datetime

def house_absences_optimized(out):
    daytimes = set(range(8, 17))  # 8 AM to 4 PM inclusive
    remove_days = set()

    # Convert datetime strings to datetime objects once and store in a list
    datetime_records = [(datetime.strptime(record[0], "%d/%m/%Y %H:%M:%S"), record[1]) for record in out]

    for i in range(len(datetime_records) - 1):
        record = datetime_records[i]
        next_record = datetime_records[i + 1]

        if record[0].hour in daytimes:
            if next_record[0].date() != record[0].date() and next_record[0].hour > 8:
                remove_days.add(record[0].date())
                remove_days.add(next_record[0].date())
                print("Remove absences",record,next_record)
    print(remove_days)
    # Filter out entries from days to be removed
    filtered_out = [record for record in out if datetime.strptime(record[0], "%d/%m/%Y %H:%M:%S").date() not in remove_days]

    return filtered_out




In [10]:
# Function to parse date strings into datetime objects
def parse_date(date_string):
    return datetime.strptime(date_string, "%d/%m/%Y %H:%M:%S")

In [11]:
MCI_patients=[1464, 1508, 1511, 1603, 1609, 1607, 1627, 1628, 1618, 1657, 1665, 1724]
clinical_df=read_clinical()
MCI_patients_df=clinical_df[clinical_df['home_id'].isin(MCI_patients)].reset_index(drop=True)
MCI_patients_df

,sub_id,home_id,study,redcap_event_name,date_screen,age,residenc,othcondx,cogothx
0,1655,1464,VA-CART,baseline_visit_arm_1,2018-04-02,70.7,1.0,NaN,MCI
1,1655,1464,VA-CART,month_12_arm_1,2019-04-26,71.7,1.0,"Neuropathy, action tremor, Ankylosing-spondyli...",aMCI - memory
2,1655,1464,VA-CART,month_24_arm_1,2020-07-02,72.9,1.0,"Neuropathy, action tremor, Ankylosing-spondyli...",aMCI - memory only
3,1710,1508,OHSU-CART,baseline_visit_arm_1,2018-06-21,75.4,2.0,Hysterectomy (takes hormone replacement),MCI
4,1710,1508,OHSU-CART,month_12_arm_1,2019-06-27,76.4,1.0,Sciatica diagnosed within the last 5/6 months ...,0.5 in memory only.
5,1714,1511,OHSU-CART,baseline_visit_arm_1,2018-06-27,67.1,1.0,NaN,NaN
6,1714,1511,OHSU-CART,month_12_arm_1,2019-06-17,68.1,1.0,NaN,0.5 in memory only.
7,1799,1603,VA-CART,baseline_visit_arm_1,2018-09-19,72.7,1.0,"Carotid Enderterectomy (2018), Abdominal Aorti...",MCI - memory
8,1799,1603,VA-CART,month_12_arm_1,2019-09-13,73.7,1.0,"Carotid endarterectomy, abdominal aortic bypas...",non-amnestic MCI - executive
9,1805,1609,VA-CART,baseline_visit_arm_1,2018-09-24,74.2,1.0,NaN,MCI


In [12]:
# clinical_df['othcondx'].unique()

In [12]:
# import pandas as pd
# physical_problems=[
#     "Neuropathy", "Staph infection", "Lumbar spinal stenosis", "Enlarged prostate", "Gout",
#     "Hysterectomy", "Ablation", "Cataracts", "Upper/lower GI problems", "Vomiting diarrhea cramps",
#     "Sciatica", "Hip revision", "Hepatitis C", "TBI (Traumatic Brain Injury)", "Degenerated vertebrae",
#     "Asthma", "Fibromyalgia", "Hip/Knee replacement", "Spinal stenosis", "Carpal tunnel",
#     "Shingles", "Vision loss spells", "Carotid endarterectomy", "Abdominal aortic bypass",
#     "Partial amputation", "Spinal cord stimulator implant", "Pain pump implant", "Arrhythmia",
#     "COPD (Chronic Obstructive Pulmonary Disease)", "GERD (Gastroesophageal Reflux Disease)",
#     "Basal squamous skin cancer", "Bradycardia", "Bladder tack", "Barrett's Esophagus", "Glaucoma",
#     "Autoimmune psoriasis", "Wet macular degeneration", "Kidney damage", "Heart murmur",
#     "Pinched nerve", "Tore ACL", "Sprained MCL", "Bulging meniscus", "Hernia repair", "Sebaceous tumor",
#     "Benign Prostatic Hyperplasia", 
#         'Chemotherapy', 'High blood pressure','Thick mucus in lungs','Postural imbalance']
    
# mental_problems=['Tinnitus (often related to auditory perception but can have psychological effects)',
#                  'HRT (Hormone replacement therapy)',
#                  'PTSD (Post-Traumatic Stress Disorder)',
#                  'SAD (Seasonal Affective Disorder)','Nerve pain']

# # Create a list of tuples containing each condition and its category
# data = [(problem, 'Physical Problems') for problem in physical_problems] + \
#        [(problem, 'Mental Problems') for problem in mental_problems]

# # Create a DataFrame from the list of tuples
# df = pd.DataFrame(data, columns=['Condition', 'Category'])
# df

In [ ]:
#remove covid data
out=remove_covid(out)

#remove days
out,new_date=remove_day_optimized(out)
# remove absence days
out=house_absences_optimized(out)

df =  pd.DataFrame (out)

In [15]:
len(new_date)

575

In [13]:
for patient in patients: 
    out=readData(patient)
    #remove covid data
    out=remove_covid(out)
    
    #remove days
    out,new_date=remove_day_optimized(out)
    # remove absence days
    out=house_absences_optimized(out)
    
    df =  pd.DataFrame (out)

 
    df.to_csv(f"{patient}_cleaned_removedays"'.csv', index=False, header=True)


11/06/2019 {'devices': {'Kitchen 1', 'Bedroom 1', 'Front Door', 'Dining Room 1', 'Other 1', 'Hallway', 'Living Room'}}
12/06/2019 {'devices': {'Kitchen 1', 'Bedroom 1', 'Front Door', 'Dining Room 1', 'Other 1', 'Hallway', 'Living Room'}}
13/06/2019 {'devices': {'Kitchen 1', 'Bedroom 1', 'Front Door', 'Dining Room 1', 'Other 1', 'Living Room', 'Hallway'}}
14/06/2019 {'devices': {'Kitchen 1', 'Bedroom 1', 'Front Door', 'Dining Room 1', 'Other 1', 'Hallway', 'Living Room'}}
29/10/2019 {'devices': {'Hallway', 'Living Room'}}
21/12/2019 {'devices': {'Back Door', 'Kitchen 1', 'Hallway'}}
02/01/2020 {'devices': {'Hallway', 'Living Room'}}
06/01/2020 {'devices': {'Kitchen 1', 'Front Door', 'Dining Room 1', 'Living Room', 'Hallway', 'Back Door'}}
14/02/2020 {'devices': {'Dining Room 1', 'Hallway', 'Living Room'}}
set()
26/12/2018 {'devices': {'Walk-in Closet 1', 'Hallway', 'Living Room'}}
06/06/2019 {'devices': {'Living Room'}}
20/12/2019 {'devices': {'Bedroom 1'}}
Remove absences (datetime.dat

14/12/2019 {'devices': {'Other Door', 'Bedroom 1', 'Walk-in Closet 1', 'Living Room', 'Hallway'}}
Remove absences (datetime.datetime(2019, 6, 25, 11, 7, 32), 'Bedroom 1') (datetime.datetime(2019, 9, 26, 14, 28, 10), 'Library 1')
{datetime.date(2019, 6, 25), datetime.date(2019, 9, 26)}
31/07/2018 {'devices': {'Kitchen 1'}}
26/12/2018 {'devices': {'Back Door', 'Other 1', 'Hallway', 'Bedroom 2'}}
03/02/2019 {'devices': {'Hallway'}}
14/01/2020 {'devices': {'Living Room', 'Kitchen 1', 'Hallway'}}
Remove absences (datetime.datetime(2018, 5, 2, 14, 4, 51), 'Bathroom') (datetime.datetime(2018, 5, 3, 11, 36, 49), 'Living Room')
Remove absences (datetime.datetime(2018, 7, 12, 11, 7, 42), 'Hallway') (datetime.datetime(2018, 8, 2, 13, 26, 56), 'Living Room')
Remove absences (datetime.datetime(2019, 2, 1, 12, 52), 'Hallway') (datetime.datetime(2019, 2, 6, 16, 50, 26), 'Back Door')
Remove absences (datetime.datetime(2019, 6, 8, 8, 5, 15), 'Living Room') (datetime.datetime(2019, 6, 28, 15, 26, 56), '

set()
13/12/2019 {'devices': {'Living Room'}}
14/12/2019 {'devices': {'Kitchen 1', 'Living Room'}}
Remove absences (datetime.datetime(2019, 7, 3, 11, 16, 34), 'Bedroom 2') (datetime.datetime(2019, 8, 8, 13, 40, 45), 'Garage Door')
{datetime.date(2019, 7, 3), datetime.date(2019, 8, 8)}
set()
Remove absences (datetime.datetime(2019, 6, 1, 16, 8), 'Hallway') (datetime.datetime(2019, 8, 7, 10, 22, 46), 'Hallway')
{datetime.date(2019, 8, 7), datetime.date(2019, 6, 1)}
14/12/2019 {'devices': {'Kitchen 1', 'Bedroom 1', 'Front Door', 'Living Room', 'Hallway'}}
Remove absences (datetime.datetime(2020, 2, 13, 9, 10, 54), 'Hallway') (datetime.datetime(2020, 2, 26, 19, 45, 36), 'Bedroom 1')
{datetime.date(2020, 2, 26), datetime.date(2020, 2, 13)}
13/12/2019 {'devices': {'Living Room', 'Hallway'}}
14/12/2019 {'devices': {'Living Room'}}
15/12/2019 {'devices': {'Hallway', 'Living Room'}}
Remove absences (datetime.datetime(2019, 4, 1, 16, 57, 51), 'Living Room') (datetime.datetime(2019, 4, 9, 15, 16,

In [14]:
def find_date(out):
    data_by_date = {}
    for entry in out:
        timestamp, location = entry
        date = timestamp.split(" ")[0]
        
        if date not in data_by_date:
            data_by_date[date] = []
    return data_by_date

In [34]:
def read_data_cleaned(patient):
    tm = pd.read_csv(f"{patient}_cleaned_removedays"'.csv')
    out=np.array(tm).tolist() 
    return out

In [35]:
# Calculate the number of days between two dates
import datetime
def days_between_dates(date_tuple):
    start_date, end_date = date_tuple
    # Parse the dates
    start_date = datetime.datetime.strptime(start_date, '%d/%m/%Y')
    end_date = datetime.datetime.strptime(end_date, '%d/%m/%Y')
    # Calculate the difference in days
    return (end_date - start_date).days


In [37]:
user_info={}
for patient in patients:
    user_info[patient]=[]
    out=read_data_cleaned(patient)
    keys = list(find_date(out).keys())
    # Retrieve the first and last key
    first_key = keys[0]  # First key
    last_key = keys[-1]  # Last key
    user_info[patient]=[len(find_date(out)),(first_key,last_key)]

#     data =user_info[patient]
#     # Calculate the number of days from the dates in the tuple
#     num_days = days_between_dates(data[1])


    print(patient,len(find_date(out)),first_key,last_key)

1135 497 13/06/2018 29/02/2020
1450 585 23/03/2018 29/02/2020
1497 495 18/06/2018 29/02/2020
1498 500 11/07/2018 29/02/2020
1504 549 10/07/2018 20/02/2020
1506 450 13/07/2018 06/02/2020
1508 440 27/07/2018 24/02/2020
1507 412 06/08/2018 29/02/2020
1511 553 18/07/2018 29/02/2020
1514 465 28/07/2018 19/11/2019
1526 484 27/08/2018 29/02/2020
1540 349 27/08/2018 29/02/2020
1590 340 17/12/2018 21/11/2019
1615 412 03/12/2018 29/02/2020
1618 149 13/08/2019 29/02/2020
1693 152 08/02/2019 29/02/2020
1464 563 25/04/2018 29/02/2020
1522 482 24/07/2018 08/02/2020
1603 321 05/12/2018 02/12/2019
1609 373 11/12/2018 29/02/2020
1610 369 28/11/2018 29/02/2020
1607 219 14/02/2019 29/02/2020
1627 335 20/12/2018 28/12/2019
1628 383 11/01/2019 29/02/2020
1657 358 15/01/2019 19/02/2020
1660 329 22/01/2019 24/02/2020
1665 273 11/01/2019 12/12/2019
1724 328 18/03/2019 29/02/2020
1542 178 18/08/2018 06/03/2019
1541 440 09/08/2018 31/01/2020
1559 434 17/08/2018 03/12/2019
1571 465 12/09/2018 25/02/2020
1572 413

In [38]:
# user_info={}
# for patient in patients:
#     user_info[patient]=[]
#     out=read_data_cleaned(patient)
#     keys = list(find_date(out).keys())
#     # Retrieve the first and last key
#     first_key = keys[0]  # First key
#     last_key = keys[-1]  # Last key
#     user_info[patient]=[len(find_date(out)),(first_key,last_key)]

#     data =user_info[patient]
#     # Calculate the number of days from the dates in the tuple
#     num_days = days_between_dates(data[1])


#     print(patient,num_days, len(find_date(out)),first_key,last_key)

In [17]:
abnormal=[1507,1627,1571,1591]

In [54]:
# filter user that have less than 6 months data
filtered_user=[]
sorted_value=[]
for key, value in user_info.items():
    if value[0]>180 and key not in abnormal:
        filtered_user.append(key)
#         print(key,value)
#         sorted_value.append(value[0])
#         if key in MCI_patients:
#             print(key,value)

1135 [497, ('13/06/2018', '29/02/2020')]
1450 [585, ('23/03/2018', '29/02/2020')]
1497 [495, ('18/06/2018', '29/02/2020')]
1498 [500, ('11/07/2018', '29/02/2020')]
1504 [549, ('10/07/2018', '20/02/2020')]
1506 [450, ('13/07/2018', '06/02/2020')]
1508 [440, ('27/07/2018', '24/02/2020')]
1511 [553, ('18/07/2018', '29/02/2020')]
1514 [465, ('28/07/2018', '19/11/2019')]
1526 [484, ('27/08/2018', '29/02/2020')]
1540 [349, ('27/08/2018', '29/02/2020')]
1590 [340, ('17/12/2018', '21/11/2019')]
1615 [412, ('03/12/2018', '29/02/2020')]
1464 [563, ('25/04/2018', '29/02/2020')]
1522 [482, ('24/07/2018', '08/02/2020')]
1603 [321, ('05/12/2018', '02/12/2019')]
1609 [373, ('11/12/2018', '29/02/2020')]
1610 [369, ('28/11/2018', '29/02/2020')]
1607 [219, ('14/02/2019', '29/02/2020')]
1628 [383, ('11/01/2019', '29/02/2020')]
1657 [358, ('15/01/2019', '19/02/2020')]
1660 [329, ('22/01/2019', '24/02/2020')]
1665 [273, ('11/01/2019', '12/12/2019')]
1724 [328, ('18/03/2019', '29/02/2020')]
1541 [440, ('09/

In [57]:
MCI_patients=[a for a in MCI_patients if a in filtered_user]
MCI_patients=[a for a in MCI_patients if a not in abnormal]
MCI_patients

[1464, 1508, 1511, 1603, 1609, 1607, 1628, 1657, 1665, 1724]

In [58]:
filtered_user=[a for a in filtered_user if a not in abnormal]

In [61]:
filtered_user.sort()

In [62]:
len(filtered_user)

35

In [63]:
print(filtered_user)

[1135, 1450, 1464, 1497, 1498, 1504, 1506, 1508, 1511, 1514, 1522, 1526, 1540, 1541, 1559, 1572, 1582, 1586, 1590, 1592, 1600, 1603, 1607, 1609, 1610, 1615, 1617, 1628, 1657, 1660, 1665, 1699, 1706, 1707, 1724]


In [64]:
# Prepare data for DataFrame
users = []
days = []
start_dates = []
end_dates = []

for key, value in user_info.items():
    if key in filtered_user:
        users.append(key)
        days.append(value[0])
        start_dates.append(value[1][0])
        end_dates.append(value[1][1])

# Create DataFrame
df = pd.DataFrame({
    'home_id': users,
    'number of days': days,
    'start date': start_dates,
    'end date': end_dates
})
df.to_csv("cleaned_days"'.csv', index=False, header=True)

In [65]:
MCI_patients_df[MCI_patients_df['home_id'].isin([1511,1607,1618,1724])]

,sub_id,home_id,study,redcap_event_name,date_screen,age,residenc,othcondx,cogothx
5,1714,1511,OHSU-CART,baseline_visit_arm_1,2018-06-27,67.1,1.0,NaN,NaN
6,1714,1511,OHSU-CART,month_12_arm_1,2019-06-17,68.1,1.0,NaN,0.5 in memory only.
11,1834,1607,VA-CART,baseline_visit_arm_1,2018-10-19,78.0,1.0,Arrhythmia,NaN
12,1834,1607,VA-CART,month_12_arm_1,2019-11-05,79.1,1.0,"COPD, Glasses, hearing aids, BPH, GERD, basal ...",non-amnestic MCI - executive
13,1834,1607,VA-CART,month_24_arm_1,2021-01-21,80.3,1.0,"COPD, BPH, Glasses, Hearing aids, arrhythmia, ...",aMCI - memory & executive
20,1856,1618,OHSU-CART,baseline_visit_arm_1,2018-11-15,77.6,2.0,NaN,0.5 in memory only.
21,1856,1618,OHSU-CART,month_12_arm_1,2019-11-13,78.6,1.0,Neuropathy in both feet,NaN
28,1970,1724,VA-CART,baseline_visit_arm_1,2019-02-25,75.1,1.0,Benign Prostatic Hyperplasia,NaN
29,1970,1724,VA-CART,month_12_arm_1,2020-02-12,76.1,1.0,"Glasses, enlarged prostate",non-aMCI - executive
30,1970,1724,VA-CART,month_24_arm_1,2021-03-19,NaN,NaN,NaN,NaN


In [ ]:
xx